# 01 — Exploratory Data Analysis: CRMLS Sold Listings

**Project:** Automated Valuation Model (AVM) — predict `ClosePrice` for California residential properties
**Data:** Monthly CRMLS sold-listing exports (`data/CRMLSSold*.csv`, Jan 2024 – Apr 2026)
**Author:** Ethan Chung — IDX Exchange Data Science, Fall 2026

**Goals for this notebook (Week 1–2):**
1. Load and combine all monthly files
2. Get a high-level overview (size, types, missingness)
3. Document what every column means (data dictionary)
4. Filter to the modeling population (residential single-family homes)
5. Look at the target (`ClosePrice`) and run basic data-quality checks

> Raw data lives in `data/` and is git-ignored — never commit CSVs.

## 1. Setup

In [ ]:
import glob
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_DIR = "../data"

# Columns that identify real people or properties — never print these in a public notebook
PII_COLS = [
    "ListAgentEmail", "ListAgentFirstName", "ListAgentLastName", "ListAgentFullName",
    "CoListAgentFirstName", "CoListAgentLastName", "BuyerAgentFirstName",
    "BuyerAgentLastName", "CoBuyerAgentFirstName", "BuyerAgentMlsId", "UnparsedAddress",
    # Not names, but together they pin down a real address/listing
    "Latitude", "Longitude", "latfilled", "lonfilled", "StreetNumberNumeric",
    "ListingId", "ListingKey", "ListingKeyNumeric",
]

## 2. Load and combine all monthly files

Each file is one month of closed sales. Some files end in `_filled` (Jan & Mar–Jul 2024, Jan 2025) —
these carry two extra columns, `latfilled` and `lonfilled`. They are **True/False flags**, not coordinates:
True means `Latitude`/`Longitude` on that row was imputed (~37% of rows in those files).
*TODO: confirm with the team how the coordinates were imputed.*

In [ ]:
files = sorted(glob.glob(os.path.join(DATA_DIR, "CRMLSSold*.csv")))
print(f"{len(files)} files found")

frames = []
for f in files:
    name = os.path.basename(f)
    tmp = pd.read_csv(f, low_memory=False)
    tmp["source_file"] = name
    tmp["is_filled_file"] = "_filled" in name
    frames.append(tmp)

raw = pd.concat(frames, ignore_index=True)
print("Combined shape:", raw.shape)

In [ ]:
# Which columns are NOT in every file?
all_cols = [set(fr.columns) for fr in frames]
common = set.intersection(*all_cols)
extra = sorted(set.union(*all_cols) - common)
print(f"{len(common)} columns in every file; extra columns in some files: {extra}")

In [ ]:
# Parse dates
for c in ["CloseDate", "ListingContractDate", "PurchaseContractDate", "ContractStatusChangeDate"]:
    raw[c] = pd.to_datetime(raw[c], errors="coerce")

raw["close_month"] = raw["CloseDate"].dt.to_period("M")
raw.groupby("close_month").size().plot(kind="bar", figsize=(12, 3), title="Closed sales per month (all property types)")
plt.ylabel("rows"); plt.tight_layout(); plt.show()

## 3. High-level overview

In [ ]:
print("Rows:", f"{len(raw):,}", "| Columns:", raw.shape[1])
print("\nDtypes:\n", raw.dtypes.value_counts())

In [ ]:
# Property types in the full data — most of the file is NOT what we're modeling
raw["PropertyType"].value_counts()

### Checks on fields assumed constant

In [ ]:
# MlsStatus should be Closed everywhere; StateOrProvince should be CA
print(raw["MlsStatus"].value_counts(dropna=False), "\n")
print(raw["StateOrProvince"].value_counts(dropna=False).head(10))
print("Non-CA rows:", (raw["StateOrProvince"] != "CA").sum())

### Duplicate listings across the full dataset

Checked on `raw` (before filtering) because the same `ListingKey` can show up in two monthly files.

In [ ]:
dups = raw[raw["ListingKey"].duplicated(keep=False)].sort_values(["ListingKey", "CloseDate"])
g = dups.groupby("ListingKey")
print(f"Rows sharing a ListingKey: {len(dups):,}  |  distinct keys: {dups['ListingKey'].nunique():,}")
print("Keys spread over more than one file:", (g["source_file"].nunique() > 1).sum())
print("Keys whose ClosePrice is identical across copies:", f"{(g['ClosePrice'].nunique() == 1).mean():.0%}")
print("\nDays between first and last CloseDate of a duplicated key:")
print((g["CloseDate"].max() - g["CloseDate"].min()).dt.days.describe(percentiles=[0.5, 0.9]))

## 4. Data dictionary

Descriptions are based on the RESO Data Dictionary field names used by CRMLS.
**Verify against the IDX metadata document** in the track channel and update anything that differs.

In [ ]:
COLUMN_DESCRIPTIONS = {
    # --- Target & prices ---
    "ClosePrice": "TARGET. Final sale price the property closed at ($).",
    "ListPrice": "Asking price at the time of sale ($). Strongly correlated with ClosePrice — beware leakage.",
    "OriginalListPrice": "First asking price when the listing went live, before any price changes ($).",
    # --- Dates & timing ---
    "CloseDate": "Date the sale closed (title transferred).",
    "ListingContractDate": "Date the listing agreement was signed / listing started.",
    "PurchaseContractDate": "Date the buyer's offer was accepted (went under contract).",
    "ContractStatusChangeDate": "Date of the most recent status change on the listing.",
    "DaysOnMarket": "Days between listing and going under contract.",
    # --- Property type ---
    "PropertyType": "Top-level category (Residential, ResidentialLease, Land, Commercial, ...).",
    "PropertySubType": "Specific type (SingleFamilyResidence, Condominium, Townhouse, ...).",
    "MlsStatus": "Listing status in the MLS — 'Closed' for every row (checked in section 3).",
    "NewConstructionYN": "True if the home is newly built.",
    "BuilderName": "Builder of the home (mostly for new construction).",
    "BusinessType": "Type of business (for business-opportunity listings only).",
    # --- Size & structure ---
    "LivingArea": "Finished living space (sq ft). Key size feature.",
    "AboveGradeFinishedArea": "Finished sq ft above ground level.",
    "BelowGradeFinishedArea": "Finished sq ft below ground (basement).",
    "BuildingAreaTotal": "Total building sq ft, including unfinished areas.",
    "BedroomsTotal": "Total number of bedrooms.",
    "BathroomsTotalInteger": "Total bathrooms, rounded to an integer.",
    "MainLevelBedrooms": "Bedrooms on the main (entry) level.",
    "Stories": "Number of stories in the home.",
    "Levels": "Text description of levels (One, Two, Multi/Split, ...).",
    "YearBuilt": "Year the home was built. Use to derive property age.",
    "Flooring": "Comma-separated list of flooring types (Carpet, Wood, Tile, ...).",
    "FireplaceYN": "True if the home has a fireplace.",
    "FireplacesTotal": "Number of fireplaces.",
    "BasementYN": "True if the home has a basement.",
    # --- Lot ---
    "LotSizeSquareFeet": "Lot size in square feet.",
    "LotSizeAcres": "Lot size in acres (should equal LotSizeSquareFeet / 43,560).",
    "LotSizeArea": "Lot size as entered by the agent (units vary).",
    "LotSizeDimensions": "Lot dimensions as text (e.g. '50x120').",
    # --- Parking ---
    "GarageSpaces": "Number of garage parking spaces.",
    "AttachedGarageYN": "True if the garage is attached to the home.",
    "ParkingTotal": "Total parking spaces (garage + other).",
    "CoveredSpaces": "Number of covered parking spaces.",
    # --- Amenities ---
    "PoolPrivateYN": "True if the property has a private pool.",
    "ViewYN": "True if the property has a view (ocean, mountain, city, ...).",
    "WaterfrontYN": "True if the property is on the waterfront.",
    # --- HOA & tax ---
    "AssociationFee": "HOA fee amount ($).",
    "AssociationFeeFrequency": "How often the HOA fee is charged (Monthly, Annually, ...).",
    "TaxAnnualAmount": "Annual property tax ($).",
    "TaxYear": "Year the tax amount applies to.",
    # --- Location ---
    "Latitude": "Latitude of the property.",
    "Longitude": "Longitude of the property.",
    "latfilled": "Flag: True if Latitude was imputed (only in _filled files).",
    "lonfilled": "Flag: True if Longitude was imputed (only in _filled files).",
    "UnparsedAddress": "Full street address as one string (PII — do not display).",
    "StreetNumberNumeric": "Street number of the address.",
    "City": "City the property is in.",
    "PostalCode": "ZIP code.",
    "CountyOrParish": "County the property is in.",
    "StateOrProvince": "State. CA for all but ~80 rows (out-of-state listings).",
    "MLSAreaMajor": "MLS-defined neighborhood/area code + name.",
    "SubdivisionName": "Name of the subdivision / tract / community.",
    # --- Schools ---
    "ElementarySchool": "Assigned elementary school.",
    "ElementarySchoolDistrict": "Elementary school district.",
    "MiddleOrJuniorSchool": "Assigned middle / junior high school.",
    "MiddleOrJuniorSchoolDistrict": "Middle school district.",
    "HighSchool": "Assigned high school.",
    "HighSchoolDistrict": "High school district.",
    # --- IDs ---
    "ListingKey": "Unique system ID for the listing record.",
    "ListingKeyNumeric": "Numeric version of ListingKey.",
    "ListingId": "MLS number shown to agents/public.",
    # --- Agents & offices (not predictive of value; some are PII) ---
    "ListAgentFirstName": "Listing agent first name (PII).",
    "ListAgentLastName": "Listing agent last name (PII).",
    "ListAgentFullName": "Listing agent full name (PII).",
    "ListAgentEmail": "Listing agent email (PII).",
    "ListAgentAOR": "Listing agent's Association of Realtors (only in some files).",
    "ListOfficeName": "Listing brokerage.",
    "CoListAgentFirstName": "Co-listing agent first name (PII).",
    "CoListAgentLastName": "Co-listing agent last name (PII).",
    "CoListOfficeName": "Co-listing brokerage.",
    "BuyerAgentFirstName": "Buyer's agent first name (PII).",
    "BuyerAgentLastName": "Buyer's agent last name (PII).",
    "BuyerAgentMlsId": "Buyer's agent MLS ID.",
    "BuyerAgentAOR": "Buyer's agent Association of Realtors (only in some files).",
    "BuyerOfficeName": "Buyer's brokerage.",
    "BuyerOfficeAOR": "Buyer's office Association of Realtors.",
    "CoBuyerAgentFirstName": "Co-buyer agent first name (PII).",
    "BuyerAgencyCompensation": "Commission offered to the buyer's agent (only in some files).",
    "BuyerAgencyCompensationType": "Unit of BuyerAgencyCompensation (% or $).",
    # --- Source system ---
    "OriginatingSystemName": "MLS system the record came from (only in some files).",
    "OriginatingSystemSubName": "Sub-MLS the record came from (only in some files).",
    # --- Added in this notebook ---
    "source_file": "(Added) CSV file the row came from.",
    "is_filled_file": "(Added) True if the row came from a *_filled.csv file.",
    "close_month": "(Added) Year-month of CloseDate.",
}

In [ ]:
def build_dictionary(df):
    rows = []
    for c in df.columns:
        s = df[c]
        example = "" if c in PII_COLS else (s.dropna().iloc[0] if s.notna().any() else "")
        rows.append({
            "column": c,
            "description": COLUMN_DESCRIPTIONS.get(c, "TODO"),
            "dtype": str(s.dtype),
            "pct_missing": round(s.isna().mean() * 100, 1),
            "n_unique": s.nunique(dropna=True),
            "example": example,
        })
    return pd.DataFrame(rows)

data_dict = build_dictionary(raw)
print("Columns still missing a description:", data_dict.loc[data_dict.description == "TODO", "column"].tolist())
data_dict

In [ ]:
# Save the dictionary so it can be shared / put in the README
os.makedirs("../docs", exist_ok=True)
data_dict.to_csv("../docs/data_dictionary.csv", index=False)

## 5. Filter to the modeling population

The AVM targets **residential single-family homes**, so we drop leases, land, commercial, condos, etc.

In [ ]:
sfr = raw[(raw["PropertyType"] == "Residential") & (raw["PropertySubType"] == "SingleFamilyResidence")].copy()
print(f"All rows: {len(raw):,}  ->  Residential SFR: {len(sfr):,}  ({len(sfr)/len(raw):.1%})")

In [ ]:
# Missingness within the modeling population
missing = (sfr.isna().mean() * 100).sort_values(ascending=False)
missing.head(40).plot(kind="barh", figsize=(8, 10), title="% missing (top 40 columns, SFR only)")
plt.gca().invert_yaxis(); plt.xlabel("% missing"); plt.tight_layout(); plt.show()

print("Columns >= 90% missing (candidates to drop):")
print(missing[missing >= 90].index.tolist())

## 6. Target: `ClosePrice`

In [ ]:
sfr["ClosePrice"].describe(percentiles=[0.005, 0.01, 0.25, 0.5, 0.75, 0.99, 0.995])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
p = sfr["ClosePrice"].clip(upper=sfr["ClosePrice"].quantile(0.99))
axes[0].hist(p, bins=100)
axes[0].set_title("ClosePrice (capped at 99th pct)")
axes[0].set_xlabel("$")

pos = sfr.loc[sfr["ClosePrice"] > 0, "ClosePrice"]
axes[1].hist(np.log10(pos), bins=100)
axes[1].set_title("log10(ClosePrice)")
axes[1].set_xlabel("log10($)")
plt.tight_layout(); plt.show()

In [ ]:
# Median price by county (top 15 by volume)
top_counties = sfr["CountyOrParish"].value_counts().head(15).index
(sfr[sfr["CountyOrParish"].isin(top_counties)]
    .groupby("CountyOrParish")["ClosePrice"]
    .agg(sales="count", median_price="median")
    .sort_values("sales", ascending=False))

In [ ]:
# Median price over time
sfr.groupby("close_month")["ClosePrice"].median().plot(figsize=(12, 3), marker="o", title="Median SFR ClosePrice by month")
plt.ylabel("$"); plt.tight_layout(); plt.show()

## 6b. Feature distributions

Size and room counts are the strongest property features, so check their shape and outliers before cleaning.
Plots are capped at the 99.5th percentile (lot size on a log scale) so a few extreme values don't flatten them.

In [ ]:
features = ["LivingArea", "BedroomsTotal", "BathroomsTotalInteger", "LotSizeSquareFeet"]
sfr[features].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

la = sfr["LivingArea"].where(sfr["LivingArea"] > 0).dropna()
axes[0, 0].hist(la.clip(upper=la.quantile(0.995)), bins=80)
axes[0, 0].set_title("LivingArea (sq ft, capped at 99.5th pct)")

for ax, col in [(axes[0, 1], "BedroomsTotal"), (axes[1, 0], "BathroomsTotalInteger")]:
    counts = sfr[col].clip(upper=8).value_counts().sort_index()
    ax.bar(counts.index, counts.values)
    ax.set_xticks(range(0, 9), [str(k) for k in range(8)] + ["8+"])
    ax.set_title(f"{col} (8+ grouped)")

lot = sfr["LotSizeSquareFeet"].where(sfr["LotSizeSquareFeet"] > 0).dropna()
axes[1, 1].hist(np.log10(lot.clip(upper=lot.quantile(0.995))), bins=80)
axes[1, 1].set_title("log10(LotSizeSquareFeet, capped at 99.5th pct)")

for ax in axes.flat:
    ax.set_ylabel("sales")
plt.tight_layout(); plt.show()

In [ ]:
print("Share of sales by bedrooms (%):")
print((sfr["BedroomsTotal"].value_counts(normalize=True).sort_index() * 100).round(1).head(10))
print("\nShare of sales by bathrooms (%):")
print((sfr["BathroomsTotalInteger"].value_counts(normalize=True).sort_index() * 100).round(1).head(10))
print(f"\nLots over 1 acre: {(sfr['LotSizeSquareFeet'] > 43_560).mean():.1%}")

**Observations**
- **LivingArea:** median 1,804 sq ft (middle 50%: 1,376–2,421), right-skewed with a long tail. Max 123,764 sq ft and some zeros are entry errors.
- **Bedrooms:** 3 (42%) and 4 (33%) dominate; 0 beds (0.1%) and values up to 45 are errors.
- **Bathrooms:** 2 (43%) and 3 (33%) dominate; max 175 is an error.
- **Lot size:** median 7,245 sq ft, extremely skewed (5.7% over 1 acre, max ≈ 2 billion sq ft) → use log scale, null absurd lots. 162 lots are ≤ 0.

## 7. Data-quality checks

In [ ]:
checks = {
    "ClosePrice <= 0": (sfr["ClosePrice"] <= 0).sum(),
    "ClosePrice < $10k (likely errors)": (sfr["ClosePrice"] < 10_000).sum(),
    "ClosePrice > $50M": (sfr["ClosePrice"] > 50_000_000).sum(),
    "LivingArea <= 0 or missing": ((sfr["LivingArea"] <= 0) | sfr["LivingArea"].isna()).sum(),
    "BedroomsTotal == 0": (sfr["BedroomsTotal"] == 0).sum(),
    "BathroomsTotalInteger == 0": (sfr["BathroomsTotalInteger"] == 0).sum(),
    "YearBuilt missing or < 1800": (sfr["YearBuilt"].isna() | (sfr["YearBuilt"] < 1800)).sum(),
    "Duplicate ListingKey": sfr["ListingKey"].duplicated().sum(),
    "Latitude/Longitude missing": sfr["Latitude"].isna().sum(),
    "Latitude outside California": (~sfr["Latitude"].between(32.3, 42.1) & sfr["Latitude"].notna()).sum(),
    "Coordinates imputed (latfilled == True)": sfr["latfilled"].astype(str).eq("True").sum(),
}
pd.Series(checks, name="count").to_frame()

In [ ]:
# Close-to-list ratio: values far from 1 usually mean a data-entry error
ratio = sfr["ClosePrice"] / sfr["ListPrice"].where(sfr["ListPrice"] > 0)
print(ratio.describe(percentiles=[0.001, 0.01, 0.5, 0.99, 0.999]))
print("Ratio < 0.5 or > 2:", ((ratio < 0.5) | (ratio > 2)).sum())

In [ ]:
# Lot size consistency: acres * 43,560 should equal square feet
both = sfr.dropna(subset=["LotSizeAcres", "LotSizeSquareFeet"])
both = both[both["LotSizeSquareFeet"] > 0]
rel_diff = (both["LotSizeAcres"] * 43_560 - both["LotSizeSquareFeet"]).abs() / both["LotSizeSquareFeet"]
print(f"Rows with both lot fields: {len(both):,}")
print(f"Match within 1%: {(rel_diff < 0.01).mean():.1%}")
print(f"Off by > 50%:   {(rel_diff > 0.5).mean():.1%}")

In [ ]:
# Price per sq ft — a sanity check that catches bad LivingArea or ClosePrice values
ppsf = sfr["ClosePrice"] / sfr["LivingArea"].where(sfr["LivingArea"] > 0)
ppsf.describe(percentiles=[0.005, 0.01, 0.5, 0.99, 0.995])

### What extreme close/list ratios look like

Most are typos in `ClosePrice` or `ListPrice` (an extra or missing zero, or a price in thousands).

In [ ]:
cols = ["ClosePrice", "ListPrice", "OriginalListPrice", "LivingArea", "CountyOrParish"]
extreme = sfr.assign(ratio=ratio)[(ratio < 0.5) | (ratio > 2)]
extreme.sort_values("ratio")[cols + ["ratio"]].iloc[np.r_[0:5, -5:0]]

## 8. Key findings

- **Size:** 615,707 closed sales across 28 monthly files (Jan 2024 – Apr 2026), 86 columns. 309,735 (50.3%) are residential single-family homes, which is the modeling population.
- **Constant fields:** `MlsStatus` is "Closed" on every row. `StateOrProvince` is CA except for ~80 rows (22 of them single-family homes) → drop those.
- **Duplicates:** 580 `ListingKey`s appear 2–4 times (1,174 rows), 264 of them among single-family homes. Mostly the same sale re-recorded in a later file with a corrected `CloseDate` (median 20 days apart); price identical in 73% of cases → keep the most recently updated record.
- **Target:** `ClosePrice` is heavily right-skewed (median ≈ $894k, mean ≈ $1.30M) → model `log(ClosePrice)`.
- **Outliers:** 31 sales above $50M; most are typos (e.g. $970M closing on a $975k list price). 235 single-family sales have a close/list ratio below 0.5 or above 2, max 1,153.85. 8 sales below $10k, 1 at $0. A close/list ratio band of [0.5, 2] catches almost all of them.
- **Missingness:** 9 columns are 100% empty for single-family homes (tax, fireplaces count, school districts, …) → drop. `BasementYN` / `WaterfrontYN` only ever record True, so missing means "no".
- **Coordinates:** 2,712 single-family rows have no latitude; a few dozen sit outside California (0,0 or sign errors). `latfilled`/`lonfilled` are flags meaning the coordinate was imputed, not replacement values.
- **Lot size:** `LotSizeAcres` and `LotSizeSquareFeet` mostly agree → keep square feet, fill from acres when missing, null lots over 1,000 acres.
- **Feature distributions:** living area, beds and baths are right-skewed with clear entry errors in the tail (123,764 sq ft, 45 beds, 175 baths); lot size is extremely skewed → log scale.
- **Leakage (excluded as features, per the IDX Best Practices doc):** `ListPrice`, `OriginalListPrice`, `DaysOnMarket` and post-listing / post-close dates. They stay in the clean table only for data-quality checks such as the close/list ratio.

## Next steps
- Cleaning rules implemented in `src/cleaning.py`, walked through in `02_preprocessing.ipynb`.
- Split by time: test = latest month; the number of prior months used for training is tuned by experiment.
- Outliers: cut ClosePrice at the 0.5th / 99.5th percentile computed on the training set only, then apply the same cutoffs to test.
- Confirm with the team how coordinates were imputed in the `_filled` files.